In [1]:
pip install openai

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from openai import OpenAI
import time
import sys

# 将当前目录加入系统路径，确保能找到我们刚才抽取的 RAG 模块
if "." not in sys.path:
    sys.path.append(".")

# 导入刚才抽取好的本地抗噪 RAG 系统
from importlib import import_module
NoiseRobustRAG = import_module("03_advanced_rag_pipeline").NoiseRobustRAG

class RAGGenerator:
# ...（这下面是你原有的 RAGGenerator 类代码，保持不变即可）...
    """
    面向事实一致性的 RAG 生成模块
    负责将检索到的高质量上下文与 User Query 融合成 Prompt，并调用 LLM 生成最终答案。
    """
    def __init__(self, api_key, base_url, model_name):
        """
        初始化 LLM 客户端
        这里默认使用 OpenAI 兼容接口，你可以轻松替换为 DeepSeek, 阿里通义千问, 智谱等。
        """
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model_name = model_name
        print(f"✅ 成功初始化生成模型: {self.model_name}")

    def build_prompt(self, query, context_list):
        """
        构建防幻觉的 System Prompt (核心提示词工程)
        对应论文中“保障事实一致性”的要求
        """
        # 将我们 Phase 3 截断后的纯净上下文拼接成字符串
        context_str = "\n".join([f"证据 {i+1}: {text}" for i, (text, score) in enumerate(context_list)])
        
        system_prompt = (
            "你是一个严谨的电商智能客服助手。\n"
            "【核心任务】：请严格基于以下提供的【真实用户评论证据】来回答用户的问题。\n"
            "【严格约束】：\n"
            "1. 你的回答必须100%忠实于提供的证据，绝不能捏造、发散或使用你的内在先验知识。\n"
            "2. 如果提供的证据中没有包含回答问题所需的信息，请直接回答“根据已有信息，无法得出结论”，绝不能强行猜测。\n"
            "3. 综合证据中的多方观点，给出客观、中立的总结。"
        )
        
        user_prompt = f"【真实用户评论证据】:\n{context_str}\n\n【用户提问】: {query}\n\n请给出你的回答："
        
        return system_prompt, user_prompt

    def generate_answer(self, query, context_list):
        """
        调用 LLM 生成答案
        """
        if not context_list:
            return "抱歉，知识库中没有检索到与您问题相关的高质量事实依据。"
            
        system_prompt, user_prompt = self.build_prompt(query, context_list)
        
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1, # 极低的 temperature 保证生成的事实一致性，拒绝发散
                max_tokens=512
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"LLM 调用失败: {e}"

# ==========================================
# 实验测试区：端到端 (End-to-End) 真实 RAG 测试
# ==========================================
if __name__ == "__main__":
    # --- ⚠️ 填入你自己的 API 配置（保持你原来的不变即可） ---
    API_KEY = "sk-44d6e7d7fee347eeabbc89410989c8c8"  # 替换为你的真实 API Key
    BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"     
    MODEL_NAME = "qwen-plus"                 
    
    # 初始化：云端生成大模型
    generator = RAGGenerator(api_key=API_KEY, base_url=BASE_URL, model_name=MODEL_NAME)
    
    # 🌟【架构改变核心点】我们要先启动并加载刚提取出来的“本地预训练 AI”（检索与抗噪）
    print("=" * 50)
    print("🤖 步骤 1：正在加载本地检索预训练模型和知识库引擎...")
    rag_system = NoiseRobustRAG(
        corpus_path="./data/processed/tablet_corpus.csv",
        bm25_path="./indices/bm25_index.pkl",
        faiss_path="./indices/faiss_index.bin"
    )
    
    # 💬 你想测试的真实问题，可以在这里随意修改：
    test_query = "这款平板屏幕质量怎么样？"

    print("=" * 50)
    print("🕵️ 步骤 2：正在调用本地双 AI（Bi-Encoder + Cross-Encoder）进行海量文本深度检索和降噪...")
    start_time_retrieval = time.time()
    
    # 真正去知识库里大海捞针！
    real_clean_context = rag_system.retrieve_context(test_query)
    
    print(f"⚡ 本地检索耗时: {time.time() - start_time_retrieval:.2f} 秒")
    print(f"🛡️ 动态截断后，共提取到 {len(real_clean_context)} 条高质量依据。")
    print("=" * 50)
    
    # 最后：把找到的高质量依据输入进 LLM
    print("🧠 步骤 3：正在将高质量查证结果，喂送给云端 LLM 大脑进行严谨思考...")
    start_time_generation = time.time()
    
    final_answer = generator.generate_answer(test_query, real_clean_context)
    
    print(f"⚡ 云端生成耗时: {time.time() - start_time_generation:.2f} 秒")
    print("=" * 50)
    print("🎉 【智能客服终端输出结果】:\n")
    print(final_answer)
    print("\n" + "=" * 50)

✅ 成功初始化生成模型: qwen-plus
🤖 步骤 1：正在加载本地检索预训练模型和知识库引擎...
正在初始化抗噪 RAG 管道...
使用计算设备: cpu


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 模型与索引加载完毕！

🕵️ 步骤 2：正在调用本地双 AI（Bi-Encoder + Cross-Encoder）进行海量文本深度检索和降噪...
⚡ 本地检索耗时: 0.62 秒
🛡️ 动态截断后，共提取到 5 条高质量依据。
🧠 步骤 3：正在将高质量查证结果，喂送给云端 LLM 大脑进行严谨思考...
⚡ 云端生成耗时: 5.04 秒
🎉 【智能客服终端输出结果】:

根据提供的用户评论证据，关于这款平板的屏幕质量存在分歧：

- 正面评价：  
  - 证据1提到“大屏，很清晰”；  
  - 证据4称“平板挺漂亮的，质量不错，系统也挺流畅”，虽未单独强调屏幕，但“漂亮”和“质量不错”可能隐含对屏幕外观和显示效果的认可；  
  - 证据3和证据5虽未直接描述屏幕，但整体高度肯定（“很不错”“很满意，超级赞”），间接支持良好体验。

- 负面评价：  
  - 证据2明确指出“屏幕不太灵敏”，并因此给出差评，认为质量差。

综上，用户对屏幕质量的反馈不一致：有用户认可其清晰度和外观，也有用户批评其触控灵敏度。因此，**屏幕质量存在个体差异或批次差异，部分用户满意，部分用户不满意**。

